In [8]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [35]:
### Helper Functions

# Function to generate a random bit using a quantum circuit
def quantum_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0) # Apply Hadamard to create a superposition
    qc.measure(0, 0) # Measure to collapse to 0 or 1 randomly
    simulator = BasicSimulator()
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(qc)
    return int(list(counts.keys())[0]) # Return 0 or 1

# Function to generate N random bits
def generate_random_bits(n):
    return [quantum_random_bit() for _ in range(n)]

# Function to encode a bit into a qubit based on a chosen basis
def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 0:
        if basis == 0: # Z-basis (rectilinear): |0>
            pass # Already in |0>
        else: # X-basis (diagonal): |+>
            qc.h(0)
    else: # bit == 1
        if basis == 0: # Z-basis (rectilinear): |1>
            qc.x(0) # Apply X to get |1>
        else: # X-basis (diagonal): |->
            qc.x(0) # Get |1>
            qc.h(0) # Apply H to get |->
    return qc # Return the circuit representing the encoded qubit

# Function to measure a qubit in a chosen basis
def measure_qubit(qc, basis):
    if basis == 0: # Z-basis
        qc.measure(0, 0)
    else: # X-basis
        qc.h(0) # Change to Z-basis for measurement
        qc.measure(0, 0)

    simulator = BasicSimulator()
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(qc)
    return int(list(counts.keys())[0]) # Return the measured bit

### Alice's Actions

In [36]:
# 1. Alice generates random bits for her raw key
num_qubits = 20 # Number of qubits/bits to send
alice_bits = generate_random_bits(num_qubits)

# 2. Alice generates random bases for encoding (0 for Z-basis, 1 for X-basis)
alice_bases = generate_random_bits(num_qubits)

print("\n--- ALICE'S PART: PREPARING AND SENDING QUBITS ---")
print(f"1. Alice's secret key bits (the random sequence she wants to establish with Bob): {alice_bits}")
print(f"2. Alice's randomly chosen bases for encoding each bit: {alice_bases} (0=Z basis, 1=X basis)")
print(f"3. Alice has prepared {num_qubits} qubits, encoding her bits in her chosen bases, and sent them through the quantum channel.")

# 3. Alice encodes her bits into qubits based on her chosen bases
alice_qubits = []
for i in range(num_qubits):
    qc = encode_qubit(alice_bits[i], alice_bases[i])
    # We create a new circuit for each qubit so they are independent
    alice_qubits.append(qc.remove_final_measurements(inplace=False)) # Remove measurement for transmission


--- ALICE'S PART: PREPARING AND SENDING QUBITS ---
1. Alice's secret key bits (the random sequence she wants to establish with Bob): [0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0]
2. Alice's randomly chosen bases for encoding each bit: [0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1] (0=Z basis, 1=X basis)
3. Alice has prepared 20 qubits, encoding her bits in her chosen bases, and sent them through the quantum channel.


### Eve's Actions (Attacker)

In [37]:
# 1. Eve intercepts Alice's qubits.

# 2. Eve generates random bases for her measurements
eve_bases = generate_random_bits(num_qubits)

print("\n--- EVE'S PART: INTERCEPTION AND RE-TRANSMISSION ---")
print(f"1. Eve's randomly chosen bases for measuring Alice's qubits: {eve_bases} (0=Z basis, 1=X basis)")

# 3. Eve measures the intercepted qubits and records her results
eve_measured_bits = []
eve_re_encoded_qubits = []

for i in range(num_qubits):
    # Eve performs a measurement on the qubit she received
    measured_bit = measure_qubit(alice_qubits[i], eve_bases[i])
    e_qc = encode_qubit(measured_bit, eve_bases[i]) # Re-encode based on her measurement and basis
    eve_measured_bits.append(measured_bit)
    eve_re_encoded_qubits.append(e_qc.remove_final_measurements(inplace=False)) # Prepare for sending to Bob

print(f"2. Eve's measured bits after intercepting Alice's qubits: {eve_measured_bits}")
print(f"3. Eve has re-encoded and sent {len(eve_re_encoded_qubits)} qubits to Bob.")


--- EVE'S PART: INTERCEPTION AND RE-TRANSMISSION ---
1. Eve's randomly chosen bases for measuring Alice's qubits: [0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1] (0=Z basis, 1=X basis)
2. Eve's measured bits after intercepting Alice's qubits: [0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0]
3. Eve has re-encoded and sent 20 qubits to Bob.


### Bob's Actions

In [38]:
# 1. Bob generates random bases for his measurements
bob_bases = generate_random_bits(num_qubits)

print("\n--- BOB'S PART: RECEIVING AND MEASURING QUBITS ---")
print(f"1. Bob's randomly chosen bases for measuring the received qubits: {bob_bases} (0=Z basis, 1=X basis)")

# 2. Bob measures the received (from Eve) qubits based on his chosen bases
bob_measured_bits = []
for i in range(num_qubits):
    measured_bit = measure_qubit(eve_re_encoded_qubits[i], bob_bases[i])
    bob_measured_bits.append(measured_bit)

print(f"2. Bob's measured bits after receiving and measuring the qubits from (potentially) Eve: {bob_measured_bits}")


--- BOB'S PART: RECEIVING AND MEASURING QUBITS ---
1. Bob's randomly chosen bases for measuring the received qubits: [0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0] (0=Z basis, 1=X basis)
2. Bob's measured bits after receiving and measuring the qubits from (potentially) Eve: [0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0]


### Key Sifting

In [39]:
# Alice and Bob publicly compare their bases and keep only the bits where bases matched
alice_sifted_key = []
bob_sifted_key = []
matching_indices = []

for i in range(num_qubits):
    if alice_bases[i] == bob_bases[i]:
        alice_sifted_key.append(alice_bits[i])
        bob_sifted_key.append(bob_measured_bits[i])
        matching_indices.append(i)

print("\n--- KEY SIFTING: PUBLIC DISCUSSION OF BASES ---")
print("1. Alice and Bob publicly compare their chosen bases.")
print("2. Only bits for which their bases matched are kept.")
print(f"3. Number of bits remaining after sifting: {len(alice_sifted_key)}")
print(f"4. Alice's sifted key (her original bits where bases matched): {alice_sifted_key}")
print(f"5. Bob's sifted key (his measured bits where bases matched): {bob_sifted_key}")


--- KEY SIFTING: PUBLIC DISCUSSION OF BASES ---
1. Alice and Bob publicly compare their chosen bases.
2. Only bits for which their bases matched are kept.
3. Number of bits remaining after sifting: 9
4. Alice's sifted key (her original bits where bases matched): [0, 0, 1, 0, 1, 0, 1, 0, 1]
5. Bob's sifted key (his measured bits where bases matched): [0, 0, 1, 1, 1, 0, 1, 0, 1]


### Error Checking

In [40]:
# Alice and Bob compare a subset of their sifted keys to detect Eve
check_key_length = min(len(alice_sifted_key), len(bob_sifted_key)) // 2 # Use half of the sifted key for checking

# Split the sifted keys into a checking portion and the final key portion
alice_checking_key = alice_sifted_key[:check_key_length]
bob_checking_key = bob_sifted_key[:check_key_length]

alice_final_key = alice_sifted_key[check_key_length:]
bob_final_key = bob_sifted_key[check_key_length:]

# Compare the checking keys
mismatches = 0
for i in range(len(alice_checking_key)):
    if alice_checking_key[i] != bob_checking_key[i]:
        mismatches += 1

error_rate = mismatches / len(alice_checking_key) if len(alice_checking_key) > 0 else 0

print(f"\n--- ERROR CHECKING AND FINAL KEY GENERATION ---")
print(f"1. Alice and Bob compare a subset of their sifted keys to detect eavesdropping.")
print(f"2. Length of the checking key: {len(alice_checking_key)}")
print(f"3. Mismatches found in the checking key: {mismatches}")
print(f"4. Quantum Bit Error Rate (QBER): {error_rate:.2%}")

# Define a threshold for detecting an attacker
threshold = 0.15 # 15% error rate is a common threshold

if error_rate > threshold:
    print(f"5. QBER ({error_rate:.2%}) is above the threshold ({threshold:.2%}). Attack detected! Key discarded.")
else:
    print(f"5. QBER ({error_rate:.2%}) is below the threshold ({threshold:.2%}). No attack detected (or Eve was lucky). Key can be used.")
    print(f"6. Alice's final shared key (after removing checking bits): {alice_final_key}")
    print(f"7. Bob's final shared key (after removing checking bits): {bob_final_key}")

# Verify if the final keys match (they should if no attack or error rate is low enough)
if alice_final_key == bob_final_key:
    print("8. Verification: Final keys match (after error checking and sifting). Perfect key established.")
else:
    print("8. Verification: Final keys DO NOT match, even after error checking. The remaining key is compromised.")


--- ERROR CHECKING AND FINAL KEY GENERATION ---
1. Alice and Bob compare a subset of their sifted keys to detect eavesdropping.
2. Length of the checking key: 4
3. Mismatches found in the checking key: 1
4. Quantum Bit Error Rate (QBER): 25.00%
5. QBER (25.00%) is above the threshold (15.00%). Attack detected! Key discarded.
8. Verification: Final keys match (after error checking and sifting). Perfect key established.
